# HW3 - Q3 [35 pts]

## Important Notices

<div class="alert alert-block alert-danger">
    WARNING: <strong>REMOVE</strong> any print statements added to cells with "#export" that are used for debugging purposes befrore submitting because they will crash the autograder in Gradescope. Any additional cells can be used for testing purposes at the bottom. 
</div>

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove any comment that says "#export" because that will crash the autograder in Gradescope. We use this comment to export your code in these cells for grading.
</div>

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> import any additional libraries into this workbook.
</div>

All instructions, code comments, etc. in this notebook **are part of the assignment instructions**. That is, if there is instructions about completing a task in this notebook, that task is not optional.  

<div class="alert alert-block alert-info">
    You <strong>must</strong> implement the following functions in this notebook to receive credit.
</div>

`user()` - 1 point

`trip_statistics()` - 3 points

`busiest_hour()` - 5 points

`most_freq_pickup_locations()` - 5 points

`avg_trip_distance_and_duration()` - 6 points

`most_freq_peak_hour_fares()` - 10 points

Each function will be auto-graded using different sets of parameters or data, to ensure that values are not hard-coded.  You may assume we will only use your code to work with data from the NYC-TLC dataset during auto-grading.

In addition, you will also submit the resulting output csv from most_freq_peak_hour_fares() as output_large.csv.

`output_large.csv` - 5 points

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remove or modify the following utility functions:
</div>

`load_data()`

`main()`

<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> remodify the below cell. It contains the function for loading data and all imports, and the function for running your code.
</div>

In [3]:
#export
from pyspark.sql.functions import *
from pyspark.sql import *

Calculation started (calculation_id=fccacc76-291e-e7df-2482-06fc24f2cd7d) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


In [10]:
#### DO NOT CHANGE ANYTHING IN THIS CELL ####

def load_data(size='small'):
    # Loads the data for this question. Do not change this function.
    # This function should only be called with the parameter 'small' or 'large'
    
    if size != 'small' and size != 'large':
        print("Invalid size parameter provided. Use only 'small' or 'large'.")
        return
    
    input_bucket = "s3://cse6242-hw3-q3"
    
    # Load Trip Data
    trips_path = '/'+size+'/yellow_tripdata*'
    trips = spark.read.csv(input_bucket + trips_path, header=True, inferSchema=True)
    
    # Load Zone Data
    zones_path = '/'+size+'/taxi*'
    zones = spark.read.csv(input_bucket + zones_path, header=True, inferSchema=True)
    
    return trips, zones
    
def main(size, bucket):
    # Runs your functions
    trips, zones = load_data(size=size)
    
    print("User:", user())
    print()
    
    print("Trip Statistics:")
    ts = trip_statistics(trips)
    ts.show()
    print()
    
    print("Busiest Hour:")
    bh = busiest_hour(trips)
    bh.show(24)
    print()
    
    print("Most Frequent Pickup Locations:")
    mfpl = most_freq_pickup_locations(trips)
    mfpl.show()
    print()
    
    print("Average Trip Distance and Duration:")
    atdd = avg_trip_distance_and_duration(trips)
    atdd.show(n=24)
    print()
    
    print("Most Frequent Peak Hour Fares:")
    mfphf = most_freq_peak_hour_fares(trips, zones)
    mfphf.show()
    mfphf.coalesce(1).write.option("header","true").mode("overwrite").csv('{}/output_{}'.format(bucket, size))

Calculation started (calculation_id=6ecacc7a-c6b9-11ec-9e84-78c58945acfc) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


# Implement the below functions for this assignment:
<div class="alert alert-block alert-danger">
    WARNING: Do <strong>NOT</strong> change any function inputs or outputs, and ensure that the dataframes your code returns align with the schema definitions commented in each function. Do <strong>NOT</strong> remove the #export comment from each of the code blocks either. This can prevent your code from being converted to a python file.
</div>

## 3.1 [1 pt] Update the `user()` function
This function should return your GT username, eg: gburdell3

In [1]:
#export
def user():
    return 'abcde'

Calculation started (calculation_id=50cacc75-e7c4-c18c-ee87-bab7cd878d4d) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.2 [3 pts] Update the `trip_statistics()` function
This function performs exploratory data analysis on the column trip_distance. Compute basic statistics (count, mean, stdev, min, max) for trip_distance. 

Example output formatting:

```
+-------+------------------+
|summary|     trip_distance|
+-------+------------------+
|  count|           xxxxxxx|
|   mean|           xxxxxxx|
| stddev|           xxxxxxx|
|    min|           xxxxxxx|
|    max|           xxxxxxx|
+-------+------------------+
```
Tip: Is there a PySpark Dataframe function you can use to solve this in a single line?

In [13]:
#export
def trip_statistics(trips):
    # Select the trip_distance column and compute the summary statistics
    trip_stats = trips.select(col("trip_distance")).describe()
    
    # Rename the column to match the desired output format
    trip_stats = trip_stats.withColumnRenamed("summary", "summary") \
                           .withColumnRenamed("trip_distance", "trip_distance")
    
    return trip_stats

Calculation started (calculation_id=92cacc7c-739a-d669-a7ec-efcc1790186b) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.3 [5 pts] Update the `busiest_hour()` function

Determine the hour of the day with the highest number of trips. Display the hour (0-23) and the corresponding trip count. 

Returns a PySpark DataFrame with a single row showing the hour with the highest trip count and the corresponding number of trips. Schema (hour, trip_count) 

Example output formatting:

```
+----+----------+
|hour|trip_count|
+----+----------+
|  xx|    xxxxxx|
+----+----------+
```

In [18]:
#export
def busiest_hour(trips):
    
    trips = trips.withColumn("tpep_pickup_datetime", col("tpep_pickup_datetime").cast("Timestamp"))
    # Extract the hour from the tpep_pickup_datetime column
    trips_with_hour = trips.withColumn("hour", hour(col("tpep_pickup_datetime")))
    
    # Group by hour and count the number of trips for each hour
    hourly_trip_counts = trips_with_hour.groupBy("hour").agg(count("*").alias("trip_count"))
    
    # Find the hour with the highest trip count
    busiest_hour_df = hourly_trip_counts.orderBy(col("trip_count").desc()).limit(1)
    
    return busiest_hour_df

Calculation started (calculation_id=34cacc82-acb8-fd13-2366-13bffbf3773e) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.4 [5 pts] Update the `most_freq_pickup_locations()` function
Top 10 Most Frequent Pickup Locations

Identify the top 10 pickup locations (by PULocationID) with the highest number of trips. Display the location IDs along with their corresponding trip counts.

Example output formatting:
```
+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         xxx|    xxxxxx|
|         xxx|    xxxxxx|
|         xxx|    xxxxxx|
|         xxx|    xxxxxx|
|         ...|    ......|
+------------+----------+
```

In [20]:
#export
def most_freq_pickup_locations(trips): 
    # Group by PULocationID and count the number of trips for each location
    pickup_counts = trips.groupBy("PULocationID").agg(count("*").alias("trip_count"))
    
    # Sort by trip_count in descending order and limit to the top 10
    top_10_pickup_locations = pickup_counts.orderBy(col("trip_count").desc()).limit(10)
    
    return top_10_pickup_locations

Calculation started (calculation_id=2ccacc84-b3ef-7bfd-7cd8-9acfe5328c7d) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.5 [6 pts] Update the `avg_trip_distance_and_duration()` function
Average Trip Distance and Duration by Hour

Calculate the average trip distance and average trip duration in minutes (i.e., divided by 60) for each hour of the day (0-23). Display the hour along with the corresponding averages. To be a valid trip, it must have a non-null timestamp and a trip distance greater than zero.

Note: You can use `unix_timestamp` to help with calculating the duration. If there are null or invalid timestamps, you will want to handle those accordingly. 

Expected Output:

A table with 24 rows showing each hour (0-23) along with the average trip distance and average trip duration for that hour.

Example output formatting:
```
+----+------------------+------------------+
|hour| avg_trip_distance| avg_trip_duration|
+----+------------------+------------------+
|   0|           xxxxxxx|           xxxxxxx|
|   1|           xxxxxxx|           xxxxxxx|
|   2|           xxxxxxx|           xxxxxxx|
|   3|           xxxxxxx|           xxxxxxx|
| ...|               ...|               ...|
|  23|           xxxxxxx|           xxxxxxx|
+----+------------------+------------------+
```

In [22]:
#export
def avg_trip_distance_and_duration(trips):
    # Filter out invalid trips (null timestamps or trip_distance <= 0)
    valid_trips = trips.filter(
        (col("tpep_pickup_datetime").isNotNull()) &
        (col("tpep_dropoff_datetime").isNotNull()) &
        (col("trip_distance") > 0)
    )
    
    # Calculate trip duration in minutes
    trips_with_duration = valid_trips.withColumn(
        "trip_duration_minutes",
        (unix_timestamp(col("tpep_dropoff_datetime")) - unix_timestamp(col("tpep_pickup_datetime"))) / 60
    )
    
    # Extract the hour from the pickup datetime
    trips_with_hour = trips_with_duration.withColumn("hour", hour(col("tpep_pickup_datetime")))
    
    # Group by hour and calculate average trip distance and average trip duration
    avg_trip_stats = trips_with_hour.groupBy("hour").agg(
        avg("trip_distance").alias("avg_trip_distance"),
        avg("trip_duration_minutes").alias("avg_trip_duration")
    )
    
    # Sort by hour to ensure the output is ordered from 0 to 23
    avg_trip_stats_sorted = avg_trip_stats.orderBy("hour")
    
    return avg_trip_stats_sorted

Calculation started (calculation_id=7acacc87-aa14-7384-05c8-e4aa28863815) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


## 3.6 [10 pts] Update the `most_freq_peak_hour_fares()` function
Top 10 Most Frequent Routes During Peak Hour 

Identify the top 10 most frequent routes (combinations of PULocationID and DOLocationID) during peak hours (7 AM - 9 AM and 4 PM - 7 PM). Peak hours can be defined as 7 <= hour < 9 and 16 <= hour < 19. Display the pickup and drop-off location ID pairs along with their trip counts and average total fare rounded to two decimal places. 

Note: A route must have a different drop off location from pickup location to be considered a valid route.

Expected Output:

A table showing the top 10 routes during peak hours with their trip counts and average total fare rounded to two decimal places. Each route is represented as a combination of PULocationID-DOLocationID and PULocationID should not be the same as DOLocationID.

Example output formatting:
```
+------------+------+------------+------+----------+--------------+
|PULocationID|PUZone|DOLocationID|DOZone|trip_count|avg_total_fare|
+------------+------+------------+------+----------+--------------+
|xxx         |xxx   |xxx         |xxx   |xxx       |xx.xx         |
|xxx         |xxx   |xxx         |xxx   |xxx       |xx.xx         |
|xxx         |xxx   |xxx         |xxx   |xxx       |xx.xx         |
|...         |...   |...         |...   |...       |...           |
+------------+------+------------+------+----------+--------------|

```

In [30]:
#export
def most_freq_peak_hour_fares(trips, zones):
    # Filter trips during peak hours (7 AM - 9 AM and 4 PM - 7 PM)
    peak_hours_trips = trips.filter(
        ((hour(col("tpep_pickup_datetime")) >= 7) & (hour(col("tpep_pickup_datetime")) < 9)) |
        ((hour(col("tpep_pickup_datetime")) >= 16) & (hour(col("tpep_pickup_datetime")) < 19))
    ).filter(col("PULocationID") != col("DOLocationID"))  # Ensure pickup and drop-off locations are different

    # Join with zones to get PUZone (pickup zone)
    trips_with_puzone = peak_hours_trips.join(
        zones.withColumnRenamed("LocationID", "PULocationID").withColumnRenamed("Zone", "PUZone"),
        on="PULocationID",
        how="left"
    )

    # Join with zones to get DOZone (drop-off zone)
    trips_with_zones = trips_with_puzone.join(
        zones.withColumnRenamed("LocationID", "DOLocationID").withColumnRenamed("Zone", "DOZone"),
        on="DOLocationID",
        how="left"
    )

    # Group by PULocationID and DOLocationID, calculate trip count and average total fare
    route_stats = trips_with_zones.groupBy(
        "PULocationID", "PUZone", "DOLocationID", "DOZone"
    ).agg(
        count("*").alias("trip_count"),
        round(avg("total_amount"), 2).alias("avg_total_fare")
    )

    # Sort by trip_count in descending order and limit to top 10
    top_10_routes = route_stats.orderBy(col("trip_count").desc()).limit(10)

    return top_10_routes

Calculation started (calculation_id=3ccacc9a-cfb4-af19-0aa8-d76659cc4e46) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


<div class="alert alert-block alert-info">
Once you have finished coding, you can export the notebook from `Notebook Explorer` by selecting your notebook and clicking `Export File` from the Actions dropdown.
</div>

#### Testing

<div class="alert alert-block alert-info">
    You may use the below cell for any additional testing you need to do, however any code implemented below will not be ran or used when grading. You can run the main function over the different sized datasets for testing your functions or you run them individually like in the examples below. To get the final output csv, you will need to run most_freq_peak_hour_fares(trips, zones) using the large dataset and write the resulting dataframe to a csv. The main function will do this for you, or you can do it yourself.
</div>

In [11]:
trips, zones = load_data('small')

Calculation started (calculation_id=dccacc7a-fe86-4c29-0549-bd0a86711d8b) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.


In [14]:
ts = trip_statistics(trips)
ts.show()

Calculation started (calculation_id=60cacc7c-8484-d0b7-7c75-2ecb2455e96e) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
+-------+------------------+
|summary|     trip_distance|
+-------+------------------+
|  count|           7667792|
|   mean|  2.80108384917053|
| stddev|3.7375294029870374|
|    min|               0.0|
|    max|             831.8|
+-------+------------------+



In [31]:
main('large', 's3://cse6242-abcde')

Calculation started (calculation_id=a0cacc9b-2acb-d5de-3783-5d2e06085974) in (session=90cacc6e-ee04-429f-7a66-0087fb8b205d). Checking calculation status...


Progress:   0%|          |elapsed time = 00:00s

Calculation completed.
User: abcde

Trip Statistics:
+-------+------------------+
|summary|     trip_distance|
+-------+------------------+
|  count|         187203269|
|   mean|2.9649251809271884|
| stddev|15.179487744870528|
|    min|              -.01|
|    max|             99.95|
+-------+------------------+


Busiest Hour:
+----+----------+
|hour|trip_count|
+----+----------+
|  18|  12112144|
+----+----------+


Most Frequent Pickup Locations:
+------------+----------+
|PULocationID|trip_count|
+------------+----------+
|         237|   7899292|
|         161|   7401958|
|         236|   7153325|
|         162|   6655847|
|         186|   6518420|
|         230|   6386265|
|          48|   5944600|
|         170|   5915604|
|         234|   5742552|
|         142|   5609771|
+------------+----------+


Average Trip Distance and Duration:
+----+------------------+------------------+
|hour| avg_trip_distance| avg_trip_duration|
+----+------------------+------------------+
|   0| 4.